In [1]:
import pandas as pd

In [2]:
enron = pd.read_csv("../data/processed/enron_clean.csv")
spamassassin =  pd.read_csv("../data/processed/spamassassin_clean.csv")

In [3]:
# Both should show exactly: ['email', 'label']
print(enron.columns.tolist())
print(spamassassin.columns.tolist())

['email', 'label']
['email', 'label']


In [4]:
# How many rows and what spam share does each have?
print("Enron:", enron.shape, "| spam share:", enron["label"].mean().round(3))
print("SpamAssassin:", spamassassin.shape, "| spam share:", spamassassin["label"].mean().round(3))

Enron: (30143, 2) | spam share: 0.483
SpamAssassin: (5811, 2) | spam share: 0.303


In [5]:
# Step 1: Tag where each row came from (temporary column, we remove it later)
enron["source"] = "enron"
spamassassin["source"] = "spamassassin"

# Step 2: Stack the two tables on top of each other
# ignore_index=True gives fresh row numbers 0, 1, 2, ... for the new table
combined = pd.concat([enron, spamassassin], ignore_index=True)
print("Combined shape:", combined.shape)

# Step 3: Check for the same email appearing in BOTH datasets
# (it would be a duplicate, and could end up in both train and test later)
print("Duplicates across datasets:", combined["email"].duplicated().sum())

# Step 4: Check for the same email with DIFFERENT labels
conflict = combined.groupby("email")["label"].nunique()
print("Conflicting emails:", (conflict > 1).sum())

Combined shape: (35954, 3)
Duplicates across datasets: 124
Conflicting emails: 0


In [6]:
# Remove conflicting emails completely output showed 0 so neeed or this code , then drop the remaining duplicates
'''conflict_emails = conflict[conflict > 1].index
combined = combined[~combined["email"].isin(conflict_emails)]'''
combined = combined.drop_duplicates(subset="email")

In [7]:
# Average length (in words) per source and per label
combined["words"] = combined["email"].str.split().str.len()
print(combined.groupby(["source", "label"])["words"].median())

# How often does the word "url" appear? (our SpamAssassin cleaning added it)
combined["has_url_token"] = combined["email"].str.contains(r"\burl\b")
print(combined.groupby("source")["has_url_token"].mean().round(3))

source        label
enron         0        131.0
              1        104.0
spamassassin  0        101.0
              1        162.0
Name: words, dtype: float64
source
enron           0.010
spamassassin    0.756
Name: has_url_token, dtype: float64


In [ ]:
'''The combined data is in good shape, but the url check found a real problem that we need to fix before training'''

In [8]:
# Pick only the SpamAssassin rows (True/False for each row)
mask = combined["source"] == "spamassassin"

# For those rows only: remove the words "url" and "emailaddr"
# \b means "whole word only", so words like "curly" are not touched
# Then collapse the extra spaces left behind and trim the ends
combined.loc[mask, "email"] = (
    combined.loc[mask, "email"]
    .str.replace(r"\b(url|emailaddr)\b", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Removing words can create new duplicates or very short emails, so redo both checks
combined["words"] = combined["email"].str.split().str.len()
combined = combined[combined["words"] >= 3]
combined = combined.drop_duplicates(subset="email")
print(combined.shape)

(35809, 5)


In [9]:
combined.head()

,email,label,source,words,has_url_token
0,subject fw weather sites this is the list of w...,0,enron,60,False
1,subject i cant believe the changma wow wow wow...,1,enron,56,False
2,subject 4 color printing special request addit...,1,enron,74,False
3,subject nobody can beat us poloponybreakaway i...,1,enron,9,False
4,subject energy extravaganza 2 weeks away energ...,0,enron,212,False


In [10]:
# Should now be close to 0 for both sources
print(combined.groupby("source")["email"].apply(lambda s: s.str.contains(r"\burl\b").mean()))

# Do leftover link pieces like "http" or "www" show up more in one source?
for word in ["http", "www", "com", "emailaddr"]:
    print(word, combined.groupby("source")["email"].apply(lambda s: s.str.contains(rf"\b{word}\b").mean()).round(3).to_dict())

source
enron           0.009706
spamassassin    0.000000
Name: email, dtype: float64
http {'enron': 0.185, 'spamassassin': 0.006}
www {'enron': 0.115, 'spamassassin': 0.003}
com {'enron': 0.275, 'spamassassin': 0.079}
emailaddr {'enron': 0.0, 'spamassassin': 0.0}


In [11]:
# This pattern matches any of these WHOLE words (\b means "word boundary"):
# the link pieces plus our two placeholder words
pattern = r"\b(http|https|www|com|url|emailaddr)\b"

# Apply it to ALL rows (no source filter this time), then tidy the spaces
combined["email"] = (
    combined["email"]
    .str.replace(pattern, " ", regex=True)    # replace each match with a space
    .str.replace(r"\s+", " ", regex=True)     # many spaces in a row -> one space
    .str.strip()                              # remove spaces at the start and end
)

# Removing words can leave very short emails or create new duplicates, so redo both checks
combined["words"] = combined["email"].str.split().str.len()
combined = combined[combined["words"] >= 3]
combined = combined.drop_duplicates(subset="email")
print(combined.shape)

(35806, 5)


In [12]:
for word in ["http", "www", "com", "url"]:
    share = combined.groupby("source")["email"].apply(
        lambda s: s.str.contains(rf"\b{word}\b").mean()   # share of emails containing the word
    )
    print(word, share.round(3).to_dict())

http {'enron': 0.0, 'spamassassin': 0.0}
www {'enron': 0.0, 'spamassassin': 0.0}
com {'enron': 0.0, 'spamassassin': 0.0}
url {'enron': 0.0, 'spamassassin': 0.0}


In [ ]:
noy yet training model lets just save the combined file 

In [13]:
# --- Step 1: Final sanity checks before saving ---
# Missing values: expect 0 for both columns
print("Missing values:", combined[["email", "label"]].isna().sum().to_dict())

# Duplicate emails: expect 0
print("Duplicate emails:", combined["email"].duplicated().sum())

# Labels should only be 0 and 1
print("Unique labels:", sorted(combined["label"].unique()))

# Final size and class balance (normalize=True gives percentages)
print("Final shape:", combined.shape)
print(combined["label"].value_counts(normalize=True).round(3))

Missing values: {'email': 0, 'label': 0}
Duplicate emails: 0
Unique labels: [np.int64(0), np.int64(1)]
Final shape: (35806, 5)
label
0    0.548
1    0.452
Name: proportion, dtype: float64


In [14]:
# --- Step 2: Shuffle the rows ---
# Right now the Enron rows and SpamAssassin rows sit in separate blocks.
# sample(frac=1) means "take 100% of the rows, in random order".
# random_state=42 makes the shuffle repeatable (same order every time you run it).
# reset_index(drop=True) gives fresh row numbers 0, 1, 2, ... and throws away the old ones.
combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [15]:
# --- Step 3: Save two versions ---
import os
os.makedirs("processed", exist_ok=True)     # create the folder if it doesn't exist

# Version 1: email + label + source
# We keep "source" so that later we can test the model on Enron-only and SpamAssassin-only emails.
combined[["email", "label", "source"]].to_csv("../data/Combined/combined_with_source.csv", index=False)

# Version 2: only the two columns the model needs
combined[["email", "label"]].to_csv("../data/Combined/combined_with_source.csv", index=False)

# index=False means "don't write the row numbers into the file".
# Without it, you get an extra "Unnamed: 0" column, like the one you had to drop in the synthetic data.
print("Saved both files.")

Saved both files.
